# Exercise 3 — Pydantic validation

**After:** [Day 3](../lessons/day3-pydantic-validation/05_theory_pydantic_models.md)

In [ ]:
import json
from datetime import date, datetime
from decimal import Decimal

from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator, model_validator

def show(r, label=""):
    print(f"{label:<40} {r.status_code}  {r.text[:130]}")

print("ready")

## ⭐ Level 1 — Read the errors

Given this model, predict which of the four payloads fail and on which field. Then run it.

```python
class Reading(BaseModel):
    station: str = Field(min_length=3)
    taken_at: datetime
    temp_c: float = Field(ge=-90, le=60)
    quality: int | None = None
```

```python
A = {"station": "260", "taken_at": "2026-09-01T12:00:00", "temp_c": 21.4}
B = {"station": "26",  "taken_at": "2026-09-01T12:00:00", "temp_c": 21.4}
C = {"station": "260", "taken_at": "yesterday",           "temp_c": 21.4}
D = {"station": "260", "taken_at": "2026-09-01T12:00:00", "temp_c": 200, "quality": None}
```

In [ ]:
# Your predictions, then the code to check them


<details>
<summary>💡 Solution</summary>

```python
class Reading(BaseModel):
    station: str = Field(min_length=3)
    taken_at: datetime
    temp_c: float = Field(ge=-90, le=60)
    quality: int | None = None

payloads = {
    "A": {"station": "260", "taken_at": "2026-09-01T12:00:00", "temp_c": 21.4},
    "B": {"station": "26",  "taken_at": "2026-09-01T12:00:00", "temp_c": 21.4},
    "C": {"station": "260", "taken_at": "yesterday",           "temp_c": 21.4},
    "D": {"station": "260", "taken_at": "2026-09-01T12:00:00", "temp_c": 200, "quality": None},
}

for name, payload in payloads.items():
    try:
        Reading.model_validate(payload)
        print(f"  ✅ {name}")
    except ValidationError as exc:
        for e in exc.errors():
            print(f"  ❌ {name}  {'.'.join(map(str, e['loc'])):<10} {e['msg']}")
```

- **A** passes.
- **B** fails on `station` — `"26"` is 2 characters, `min_length=3`.
- **C** fails on `taken_at` — `"yesterday"` is not a datetime. Pydantic parses ISO 8601, not English.
- **D** fails on `temp_c` — `200` exceeds `le=60`. `quality: None` is **fine**, because the type is
  `int | None`.

Note that `station` is a `str`, so `"260"` stays a string. Had it been `int`, `"260"` would be
coerced to `260` — and `"26"` would then pass, because `min_length` doesn't apply to integers. Choose
the type before you choose the constraint.
</details>

## ⭐⭐ Level 2 — Normalise and cross-validate

Write a `SubscriptionIn` model where:

- `email` is lowercased and stripped
- `city` is title-cased and stripped, and cannot be blank
- `min_temp_c` and `max_temp_c` are both optional, but if **both** are given, min must be below max
- `channels` defaults to `["email"]` and may only contain `"email"` or `"sms"`

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
from typing import Literal

class SubscriptionIn(BaseModel):
    email: str
    city: str
    min_temp_c: float | None = None
    max_temp_c: float | None = None
    channels: list[Literal["email", "sms"]] = ["email"]

    @field_validator("email")
    @classmethod
    def tidy_email(cls, v: str) -> str:
        return v.strip().lower()

    @field_validator("city")
    @classmethod
    def tidy_city(cls, v: str) -> str:
        v = v.strip()
        if not v:
            raise ValueError("city cannot be blank")
        return v.title()

    @model_validator(mode="after")
    def check_range(self):
        if (self.min_temp_c is not None and self.max_temp_c is not None
                and self.min_temp_c >= self.max_temp_c):
            raise ValueError("min_temp_c must be below max_temp_c")
        return self          # model validators return the MODEL

s = SubscriptionIn(email="  Rasoul@EXAMPLE.com ", city="  utrecht ", max_temp_c=30)
print(s)

for bad, label in [
    ({"email": "a@b.c", "city": "   "}, "blank city"),
    ({"email": "a@b.c", "city": "X", "min_temp_c": 30, "max_temp_c": 10}, "reversed range"),
    ({"email": "a@b.c", "city": "X", "channels": ["pigeon"]}, "bad channel"),
]:
    try:
        SubscriptionIn.model_validate(bad)
        print(f"  ✅ {label} (unexpected!)")
    except ValidationError as exc:
        print(f"  ❌ {label:<16} {exc.errors()[0]['msg'][:60]}")
```

Three techniques, three jobs:

- **`field_validator`** sees one field. Use it to clean and normalise — its **return value replaces
  the field**, so cleaning happens once, at the door.
- **`model_validator(mode="after")`** sees the whole object, once every field has been parsed. That's
  the only place a rule about *two* fields can live. It returns `self`, not a value.
- **`Literal["email", "sms"]`** constrains without any validator at all, and publishes an enumerated
  schema into `/docs`.

`list[Literal[...]]` as a mutable default is safe here — Pydantic deep-copies defaults per instance.
In a plain function signature, `def f(x=[])` is still the classic bug.
</details>

## ⭐⭐⭐ Level 3 — Prove the allow-list, and break it deliberately

1. Build an endpoint returning a dict with **six** keys behind a `response_model` declaring **three**.
   Assert the other three cannot be seen.
2. Then add a field to the model that the handler never returns, and explain the status code you get
   and **why it is the right one**.
3. Finally, show that a `Decimal` and a `date` from a database row serialise correctly through the
   model but not through `json.dumps`.

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
app = FastAPI()

ROW = {"id": 1, "email": "a@b.c", "name": "Ann",
       "password_hash": "$2b$SECRET", "ssn": "123", "internal": "flagged"}

class Out3(BaseModel):
    id: int
    email: str
    name: str

@app.get("/u", response_model=Out3)
def u():
    return ROW

body = TestClient(app).get("/u").json()
assert set(body) == {"id", "email", "name"}
for leaked in ("password_hash", "ssn", "internal"):
    assert leaked not in body
print("1. nothing leaked:", body)

# 2. promise a field we don't produce
class Out4(Out3):
    phone: str

@app.get("/u2", response_model=Out4)
def u2():
    return ROW

r = TestClient(app, raise_server_exceptions=False).get("/u2")
print("2. status:", r.status_code)

# 3. boundary conversion
class RowOut(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    city: str
    obs_date: date
    temp_max_c: float

db_row = {"city": "Utrecht", "obs_date": date(2026, 9, 1),
          "temp_max_c": Decimal("21.40"), "loaded_at": datetime.now()}

@app.get("/w", response_model=RowOut)
def w():
    return db_row

try:
    json.dumps(db_row)
except TypeError as exc:
    print("3. json.dumps        ->", exc)
print("3. through the model ->", TestClient(app).get("/w").json())
```

**Part 2 gives a `500`, and that is correct.** A `422` would mean *the caller* sent something wrong;
here the caller did nothing at all. **We** promised a `phone` field and failed to produce it, which is
a server bug. `response_model` validates our own output, so this fails loudly in development instead
of shipping a malformed payload that a client discovers six months later.

**Part 3** shows the boundary earning its keep: `Decimal` and `date` both break `json.dumps`, and both
pass through the model as `21.4` and `"2026-09-01"`. `loaded_at` vanishes because it isn't declared —
the allow-list again.
</details>